# Notebook 01 — Data Exploration

**Question:** what does healthy vibration look like in this dataset, what does degradation
look like, and does the loading pipeline reproduce both?

Covers all three runs: the defect frequencies the rig geometry predicts, RMS and kurtosis
over time for every bearing, and how the bearing that fails separates from the three that
do not.

Dataset facts — source, channel counts per run, the Set 3 file-count defect, the column
dictionary — are in [`data/README.md`](../data/README.md) and nowhere else.

---

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

from nasa_bearing_anomaly.config import BEARING_PARAMS, FIGURES_DIR, TEST_CONFIG
from nasa_bearing_anomaly.loading import load_processed, load_test, save_processed
from nasa_bearing_anomaly.physics import compute_defect_frequencies

print("Imports OK")

## 1. Bearing Physics: Defect Frequencies

Before touching the data, let's compute the physical frequencies we expect to see when a bearing is damaged.

In [ ]:
freqs = compute_defect_frequencies(BEARING_PARAMS)
print("IMS Test Rig — Bearing Defect Frequencies")
print(f"  RPM: {BEARING_PARAMS['rpm']}")
print(f"  Shaft frequency: {BEARING_PARAMS['rpm'] / 60:.2f} Hz")
print()
for name, hz in freqs.items():
    print(f"  {name:12s}: {hz:7.3f} Hz")
print()
print("These are the frequencies we will monitor in the FFT spectrum.")

## 2. Load Data

If you haven't run the loader yet:
```bash
python -m nasa_bearing_anomaly.loading --test all
```

Or run it here:

In [ ]:
# ── Load all three tests ──
dfs = {}
for test_id in [1, 2, 3]:
    try:
        dfs[test_id] = load_processed(test_id)
        print(f"Test {test_id}: {len(dfs[test_id])} samples, {len(dfs[test_id].columns)} columns")
    except FileNotFoundError:
        print(f"Test {test_id}: Not found — loading from raw files...")
        df = load_test(test_id)
        save_processed(df, test_id)
        dfs[test_id] = df

In [ ]:
# Inspect Test 1 structure
print("Test 1 — first 5 rows:")
dfs[1].head()

In [ ]:
print("Test 1 — column list:")
for col in dfs[1].columns:
    print(f"  {col}")

## 3. RMS Trends — All Tests, All Bearings

RMS (Root Mean Square) is the most direct measure of overall vibration energy.
A healthy bearing has stable, low RMS. As it degrades, RMS rises — sometimes suddenly, sometimes gradually.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(15, 12))
bearing_colors = ["#58a6ff", "#3fb950", "#d29922", "#bc8cff"]

for ax, (test_id, df) in zip(axes, dfs.items(), strict=True):
    config = TEST_CONFIG[test_id]
    for i, bearing in enumerate(["Bearing1", "Bearing2", "Bearing3", "Bearing4"]):
        col = f"{bearing}_ch1_rms"
        if col in df.columns:
            label = f"{bearing}" + (" ← FAILED" if bearing == config["failed_bearing"] else "")
            lw = 1.5 if bearing == config["failed_bearing"] else 0.8
            ax.plot(df.index, df[col], color=bearing_colors[i], linewidth=lw, label=label)

    ax.set_title(
        f"Test {test_id} — {config['failed_bearing']} | {config['failure_mode']}", fontweight="bold"
    )
    ax.set_ylabel("RMS (g)")
    ax.legend(fontsize=8, loc="upper left")
    ax.grid(True, alpha=0.4)

axes[-1].set_xlabel("File Index (Time →)")
plt.suptitle(
    "NASA IMS Bearing Dataset — RMS Vibration Overview (All Tests)",
    fontsize=14,
    fontweight="bold",
    y=1.01,
)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "eda_rms_all_tests.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Kurtosis Analysis

**Why kurtosis?**

Kurtosis measures how heavy a distribution's tails are — how much of the signal's energy
sits in rare, large excursions rather than in the bulk. A healthy bearing is broadband
noise, close to Gaussian. When a race spalls, every rolling element passing the defect
produces an impact, and those impacts are exactly the rare large excursions kurtosis
responds to. It can move before RMS does, because a handful of impulses barely shifts the
total energy.

**These columns hold *excess* kurtosis, so Gaussian reads 0, not 3.** The computation
subtracts 3 (Fisher definition). Bearing-diagnostics papers normally quote the non-excess
value, which makes every published threshold 3 too high for this column:

| State | As published (non-excess) | This column (excess) |
| ----- | ------------------------- | -------------------- |
| Healthy, near-Gaussian | 3 | **0** |
| Early fault | 5 | **2** |
| Severe fault | 10 | **7** |

Getting this backwards puts the "healthy" reference line right where the failure begins.
Measured values and the convention are documented in [`data/README.md`](../data/README.md).

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(15, 10), sharex=False)

# Reference lines in excess kurtosis: the usual 3 / 5 / 10 thresholds minus 3,
# because the _kurt columns already subtract the Gaussian value.
for ax, (test_id, df) in zip(axes, dfs.items(), strict=True):
    config = TEST_CONFIG[test_id]
    failed = config["failed_bearing"]
    col = f"{failed}_ch1_kurt"

    if col in df.columns:
        ax.plot(df.index, df[col], color="#d29922", linewidth=0.9)
        ax.fill_between(df.index, df[col], 2, where=df[col] > 2, alpha=0.2, color="#f85149")

    ax.axhline(y=0, color="#3fb950", linestyle="--", alpha=0.7, label="Gaussian (0)")
    ax.axhline(y=2, color="#d29922", linestyle="--", alpha=0.7, label="Early fault (2)")
    ax.axhline(y=7, color="#f85149", linestyle="--", alpha=0.7, label="Severe fault (7)")

    ax.set_title(
        f"Test {test_id} — {failed} Kurtosis ({config['failure_mode']})", fontweight="bold"
    )
    ax.set_ylabel("Excess kurtosis")
    ax.legend(fontsize=7, loc="upper left")
    ax.grid(True, alpha=0.4)

plt.suptitle("Kurtosis Evolution — Bearing Fault Indicator", fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "eda_kurtosis_all_tests.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Statistical Summary

Compare early-stage vs late-stage statistics for the failed bearing.

In [ ]:
for test_id, df in dfs.items():
    config = TEST_CONFIG[test_id]
    failed = config["failed_bearing"]
    rms_col = f"{failed}_ch1_rms"
    kurt_col = f"{failed}_ch1_kurt"

    n = len(df)
    early = df.iloc[: int(n * 0.2)]
    late = df.iloc[int(n * 0.8) :]

    print(f"\n── Test {test_id} | {failed} ──────────────────")
    print(f"{'Metric':<20} {'Early (0-20%)':>15} {'Late (80-100%)': >15}")
    print("-" * 52)
    if rms_col in df.columns:
        print(f"{'RMS mean':<20} {early[rms_col].mean():>15.4f} {late[rms_col].mean():>15.4f}")
        print(f"{'RMS max':<20}  {early[rms_col].max():>15.4f} {late[rms_col].max():>15.4f}")
    if kurt_col in df.columns:
        print(
            f"{'Kurtosis mean':<20} {early[kurt_col].mean():>15.2f} {late[kurt_col].mean():>15.2f}"
        )
        print(f"{'Kurtosis max':<20}  {early[kurt_col].max():>15.2f} {late[kurt_col].max():>15.2f}")

## 6. Correlation Heatmap

Understand which features are correlated and might be redundant.

In [ ]:
# Correlation heatmap for Test 2 (shorter, cleaner failure)
df2 = dfs[2]
key_cols = [c for c in df2.columns if "_rms" in c or "_kurt" in c]

fig, ax = plt.subplots(figsize=(12, 10))
corr = df2[key_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr,
    mask=mask,
    cmap="RdBu_r",
    vmin=-1,
    vmax=1,
    annot=True,
    fmt=".2f",
    linewidths=0.5,
    annot_kws={"size": 7},
    ax=ax,
)
ax.set_title(
    "Test 2 — Feature Correlation Matrix (RMS + Kurtosis)", fontsize=12, fontweight="bold", pad=15
)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "eda_correlation_test2.png", dpi=150, bbox_inches="tight")
plt.show()

---
## Summary

From this exploration we confirm:

1. **Test 1** — Bearing3 shows a dramatic RMS spike in the final ~300 files
2. **Test 2** — Bearing1 shows rapid early degradation followed by catastrophic failure
3. **Test 3** — More gradual, multi-phase degradation requiring more sensitive detection
4. **Kurtosis** is a reliable early indicator — rising well before RMS shows visible change

**Next:** Feature engineering → Notebook 02